# **1. RAG V1 Environment Initialisation**


## **1.1. Load RAG Dependencies**

### **Install All Required Libraries**

In [1]:
# =============================================================================
# Retriever Implementation
# INSTALL DEPENDENCIES
# =============================================================================

# Install the libraries required for telecom RAG data acquisition,
# document processing, embeddings and vector retrieval.
#
# The current runtime is CPU-based because inference is not yet being
# performed. GPU-specific acceleration can be enabled later when required.

!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    huggingface_hub \
    safetensors \
    sentencepiece \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    python-docx \
    python-pptx \
    beautifulsoup4 \
    pyarrow \
    tqdm

print("All required Module 2 libraries installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 21.0 MB/s eta 0:00:00
All required Module 2 libraries installed successfully.


### **Import All Required Library**

In [2]:
# =============================================================================
# Retriever Implementation
# IMPORT REQUIRED LIBRARIES
# =============================================================================

# ---------------------------------------------------------------------------
# Core scientific stack
# ---------------------------------------------------------------------------
import numpy as np
import scipy
import pandas as pd

# ---------------------------------------------------------------------------
# Standard Python libraries
# ---------------------------------------------------------------------------
import os
import gc
import json
import shutil
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# Progress monitoring
# ---------------------------------------------------------------------------
from tqdm.auto import tqdm

# Data / document processing
# ---------------------------------------------------------------------------
import pyarrow
import pyarrow.parquet as pq
from docx import Document
from bs4 import BeautifulSoup
from pypdf import PdfReader

# ---------------------------------------------------------------------------
# HTTP / source acquisition
# ---------------------------------------------------------------------------
import requests

# ---------------------------------------------------------------------------
# PyTorch
# ---------------------------------------------------------------------------
import torch

# ---------------------------------------------------------------------------
# Hugging Face / LLM inference
# ---------------------------------------------------------------------------
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

from huggingface_hub import login, HfApi, snapshot_download

# ---------------------------------------------------------------------------
# Embeddings
# ---------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------------------------
# Vector similarity search
# ---------------------------------------------------------------------------
import faiss

# ---------------------------------------------------------------------------
# PDF document processing
# ---------------------------------------------------------------------------
from pypdf import PdfReader

print("All required libraries imported successfully.")

# Display key package versions for reproducibility
print("\nPackage Versions")
print("-" * 40)
print(f"NumPy           : {np.__version__}")
print(f"SciPy           : {scipy.__version__}")
print(f"Pandas          : {pd.__version__}")
print(f"PyTorch         : {torch.__version__}")
print(f"FAISS           : {faiss.__version__}")
print(f"PyArrow         : {pyarrow.__version__}")

All required libraries imported successfully.

Package Versions
----------------------------------------
NumPy           : 2.1.3
SciPy           : 1.16.3
Pandas          : 2.2.3
PyTorch         : 2.11.0+cu128
FAISS           : 1.15.0
PyArrow         : 18.1.0


### **Kaggle Login**

In [3]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


### **Dataset Import**

In [4]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

cliffordimaguezegie_telecom_chunks_path = kagglehub.dataset_download('cliffordimaguezegie/telecom-reconciled-chunks')
cliffordimaguezegie_telecom_bge_m3_embeddings_path = kagglehub.dataset_download('cliffordimaguezegie/telecom-bge-m3-embeddings')
cliffordimaguezegie_telecom_bge_m3_faiss_path = kagglehub.dataset_download('cliffordimaguezegie/telecom-bge-m3-faiss')

print('Data source import complete.')


100%|██████████| 602M/602M [00:30<00:00, 20.7MB/s]

Extracting files...


100%|██████████| 2.50G/2.50G [02:07<00:00, 21.1MB/s]

Extracting files...


100%|██████████| 3.35G/3.35G [02:54<00:00, 20.6MB/s]

Extracting files...


Data source import complete.


In [5]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

cliffordimaguezegie_telecom_benchmark_path = kagglehub.dataset_download('cliffordimaguezegie/benchmark')

print('Data source import complete.')


100%|██████████| 21.9k/21.9k [00:00<00:00, 25.1MB/s]

Extracting files...
Data source import complete.


In [6]:
print("Chunks:")
print(cliffordimaguezegie_telecom_chunks_path)

print("\nEmbeddings:")
print(cliffordimaguezegie_telecom_bge_m3_embeddings_path)

print("\nFAISS:")
print(cliffordimaguezegie_telecom_bge_m3_faiss_path)

print("\nQUESTIONS:")
print(cliffordimaguezegie_telecom_benchmark_path)




Chunks:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-reconciled-chunks/versions/1

Embeddings:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-bge-m3-embeddings/versions/1

FAISS:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-bge-m3-faiss/versions/1

QUESTIONS:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/benchmark/versions/1


In [7]:
from pathlib import Path

CHUNK_DIR = Path(
    cliffordimaguezegie_telecom_chunks_path
)

EMBED_DIR = Path(
    cliffordimaguezegie_telecom_bge_m3_embeddings_path
)

FAISS_DIR = Path(
    cliffordimaguezegie_telecom_bge_m3_faiss_path
)

BENCHMARK_DIR = Path(
    cliffordimaguezegie_telecom_benchmark_path
)

print("Chunks    :", CHUNK_DIR)
print("Embeddings:", EMBED_DIR)
print("FAISS     :", FAISS_DIR)
print("BENCHMARK :", BENCHMARK_DIR)

Chunks    : /root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-reconciled-chunks/versions/1
Embeddings: /root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-bge-m3-embeddings/versions/1
FAISS     : /root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-bge-m3-faiss/versions/1
BENCHMARK : /root/.cache/kagglehub/datasets/cliffordimaguezegie/benchmark/versions/1


In [8]:
# =============================================================================
# RAG V1 — INSPECT BENCHMARK DATASET
# =============================================================================

print("=" * 90)
print("RAG V1 — BENCHMARK DATASET CONTENTS")
print("=" * 90)

for file_path in sorted(
    BENCHMARK_DIR.rglob("*")
):

    if file_path.is_file():

        print(
            file_path.relative_to(
                BENCHMARK_DIR
            )
        )

print("=" * 90)

RAG V1 — BENCHMARK DATASET CONTENTS
track1_20_questions.json
track2_final_32_questions.json


In [9]:
# =============================================================================
# RAG V1 — LOAD BENCHMARK QUESTION BANKS
# =============================================================================

import json


TRACK1_FILE = (
    BENCHMARK_DIR
    / "track1_20_questions.json"
)

TRACK2_FILE = (
    BENCHMARK_DIR
    / "track2_final_32_questions.json"
)


# =============================================================================
# LOAD TRACK 1
# =============================================================================

with open(
    TRACK1_FILE,
    "r",
    encoding="utf-8",
) as file:

    benchmark_questions = json.load(
        file
    )


# =============================================================================
# LOAD TRACK 2
# =============================================================================

with open(
    TRACK2_FILE,
    "r",
    encoding="utf-8",
) as file:

    benchmark_questions_track2 = json.load(
        file
    )


# =============================================================================
# VALIDATION
# =============================================================================

print("=" * 90)
print("RAG V1 — BENCHMARK QUESTION BANKS LOADED")
print("=" * 90)

print("\nTRACK 1")
print("-" * 60)

print(
    f"Questions : "
    f"{len(benchmark_questions)}"
)

print(
    f"First ID  : "
    f"{benchmark_questions[0]['id']}"
)

print(
    f"Last ID   : "
    f"{benchmark_questions[-1]['id']}"
)


print("\nTRACK 2")
print("-" * 60)

print(
    f"Questions : "
    f"{len(benchmark_questions_track2)}"
)

print(
    f"First ID  : "
    f"{benchmark_questions_track2[0]['evaluation_id']}"
)

print(
    f"Last ID   : "
    f"{benchmark_questions_track2[-1]['evaluation_id']}"
)


# =============================================================================
# COUNT CHECKS
# =============================================================================

if len(benchmark_questions) != 20:

    raise RuntimeError(
        f"Track 1 expected 20 questions, "
        f"found {len(benchmark_questions)}."
    )


if len(benchmark_questions_track2) != 32:

    raise RuntimeError(
        f"Track 2 expected 32 questions, "
        f"found {len(benchmark_questions_track2)}."
    )


print("\n" + "=" * 90)
print("TRACK 1 + TRACK 2 QUESTION BANKS VALIDATED")
print("=" * 90)

RAG V1 — BENCHMARK QUESTION BANKS LOADED

TRACK 1
------------------------------------------------------------
Questions : 20
First ID  : Q01
Last ID   : Q20

TRACK 2
------------------------------------------------------------
Questions : 32
First ID  : T2-01
Last ID   : T2-32

TRACK 1 + TRACK 2 QUESTION BANKS VALIDATED


## **1.2. Load Retriever V1**

### **Load Retriever**

In [10]:
# =============================================================================
# RETRIEVER V1 — STANDARD IN-MEMORY IMPLEMENTATION
# =============================================================================

import json
import time
from pathlib import Path

import faiss
import numpy as np
import torch

from sentence_transformers import SentenceTransformer


# =============================================================================
# RETRIEVER V1 CONFIGURATION
# =============================================================================

MODEL_NAME = "BAAI/bge-m3"

MAX_SEQ_LENGTH = 1280

DEFAULT_K = 7

EXPECTED_VECTORS = 1_506_367

EXPECTED_DIMENSION = 1024

RETRIEVER_VERSION = "V1"


# =============================================================================
# RUNTIME OBJECTS
# =============================================================================

faiss_index = None

query_encoder = None

vector_metadata = None

chunk_text_map = None

faiss_manifest = None


# =============================================================================
# INITIALIZE RETRIEVER
# =============================================================================

def initialize_retriever(
    faiss_dir,
    chunk_dir,
    device="cuda",
):
    """
    Initialize Retriever V1.

    Loads:
        - FAISS index
        - BGE-M3 query encoder
        - Vector metadata
        - Chunk text
    """

    global faiss_index
    global query_encoder
    global vector_metadata
    global chunk_text_map
    global faiss_manifest


    # =========================================================================
    # VALIDATE DEVICE
    # =========================================================================

    if device.startswith("cuda"):

        if not torch.cuda.is_available():

            raise RuntimeError(
                "CUDA GPU is required for Retriever V1."
            )


    # =========================================================================
    # RESOLVE PATHS
    # =========================================================================

    faiss_dir = Path(
        faiss_dir
    )

    chunk_dir = Path(
        chunk_dir
    )


    # =========================================================================
    # FAISS ARTIFACTS
    # =========================================================================

    faiss_index_path = (
        faiss_dir
        / "faiss_index_flat_ip.index"
    )

    faiss_manifest_path = (
        faiss_dir
        / "faiss_manifest.json"
    )


    for path in [
        faiss_dir,
        chunk_dir,
        faiss_index_path,
        faiss_manifest_path,
    ]:

        if not path.exists():

            raise FileNotFoundError(
                f"Required Retriever V1 resource not found:\n"
                f"{path}"
            )


    # =========================================================================
    # LOAD FAISS
    # =========================================================================

    faiss_index = faiss.read_index(
        str(faiss_index_path)
    )


    if faiss_index.ntotal != EXPECTED_VECTORS:

        raise RuntimeError(
            f"Unexpected FAISS vector count: "
            f"{faiss_index.ntotal:,}"
        )


    if faiss_index.d != EXPECTED_DIMENSION:

        raise RuntimeError(
            f"Unexpected FAISS dimension: "
            f"{faiss_index.d}"
        )


    # =========================================================================
    # LOAD FAISS MANIFEST
    # =========================================================================

    with open(
        faiss_manifest_path,
        "r",
        encoding="utf-8",
    ) as file:

        faiss_manifest = json.load(
            file
        )


    # =========================================================================
    # LOAD BGE-M3
    # =========================================================================

    query_encoder = SentenceTransformer(
        MODEL_NAME,
        device=device,
    )

    query_encoder.max_seq_length = (
        MAX_SEQ_LENGTH
    )

    query_encoder.half()

    query_encoder.eval()


    # =========================================================================
    # LOAD VECTOR METADATA
    # =========================================================================

    vector_mapping_path = (
        faiss_dir
        / "vector_mapping.jsonl"
    )

    if not vector_mapping_path.exists():

        raise FileNotFoundError(
            f"Vector mapping not found:\n"
            f"{vector_mapping_path}"
        )


    vector_metadata = {}


    with open(
        vector_mapping_path,
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:

            if not line.strip():
                continue

            record = json.loads(
                line
            )

            vector_id = int(
                record["vector_id"]
            )

            vector_metadata[
                vector_id
            ] = record


    if len(vector_metadata) != EXPECTED_VECTORS:

        raise RuntimeError(
            f"Expected {EXPECTED_VECTORS:,} "
            f"metadata records, found "
            f"{len(vector_metadata):,}"
        )


    # =========================================================================
    # LOAD CHUNK TEXT
    # =========================================================================

    chunk_shards = sorted(
        chunk_dir.glob(
            "chunks_*.jsonl"
        )
    )


    if not chunk_shards:

        raise FileNotFoundError(
            "No chunk shards found in:\n"
            f"{chunk_dir}"
        )


    chunk_text_map = {}


    for shard_path in chunk_shards:

        with open(
            shard_path,
            "r",
            encoding="utf-8",
        ) as file:

            for line in file:

                if not line.strip():
                    continue

                record = json.loads(
                    line
                )

                chunk_id = record.get(
                    "chunk_id"
                )

                if chunk_id is not None:

                    chunk_text_map[
                        chunk_id
                    ] = record["text"]


    if len(chunk_text_map) != EXPECTED_VECTORS:

        raise RuntimeError(
            f"Expected {EXPECTED_VECTORS:,} "
            f"chunk texts, found "
            f"{len(chunk_text_map):,}"
        )


    # =========================================================================
    # RETURN STATUS
    # =========================================================================

    return {
        "version": RETRIEVER_VERSION,
        "model": MODEL_NAME,
        "device": device,
        "embedding_dimension":
            EXPECTED_DIMENSION,
        "faiss_vectors":
            int(faiss_index.ntotal),
        "faiss_index":
            faiss_manifest.get(
                "index_type"
            ),
        "similarity":
            faiss_manifest.get(
                "similarity"
            ),
        "metadata_rows":
            len(vector_metadata),
        "chunk_rows":
            len(chunk_text_map),
        "default_k":
            DEFAULT_K,
    }


# =============================================================================
# RETRIEVE
# =============================================================================

def retrieve(
    query: str,
    k: int = DEFAULT_K,
):
    """
    Execute Retriever V1.
    """

    global faiss_index
    global query_encoder
    global vector_metadata
    global chunk_text_map


    # =========================================================================
    # VALIDATE INITIALIZATION
    # =========================================================================

    if (
        faiss_index is None
        or query_encoder is None
        or vector_metadata is None
        or chunk_text_map is None
    ):

        raise RuntimeError(
            "Retriever V1 is not initialized. "
            "Call initialize_retriever() first."
        )


    # =========================================================================
    # INPUT VALIDATION
    # =========================================================================

    if not isinstance(
        query,
        str,
    ):

        raise TypeError(
            "query must be a string."
        )


    query = query.strip()


    if not query:

        raise ValueError(
            "query cannot be empty."
        )


    if not isinstance(
        k,
        int,
    ):

        raise TypeError(
            "k must be an integer."
        )


    if k <= 0:

        raise ValueError(
            "k must be greater than zero."
        )


    # =========================================================================
    # QUERY EMBEDDING
    # =========================================================================

    embedding_start = time.time()


    query_embedding = (
        query_encoder.encode(
            [query],
            batch_size=1,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    )


    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32,
    )


    query_embedding /= np.linalg.norm(
        query_embedding,
        axis=1,
        keepdims=True,
    )


    embedding_time = (
        time.time()
        - embedding_start
    )


    # =========================================================================
    # FAISS SEARCH
    # =========================================================================

    search_start = time.time()


    scores, vector_ids = (
        faiss_index.search(
            query_embedding,
            k,
        )
    )


    search_time = (
        time.time()
        - search_start
    )


    # =========================================================================
    # RESULT RESOLUTION
    # =========================================================================

    results = []


    for rank, (
        vector_id,
        score,
    ) in enumerate(
        zip(
            vector_ids[0],
            scores[0],
        ),
        start=1,
    ):

        vector_id = int(
            vector_id
        )


        if vector_id < 0:
            continue


        metadata = vector_metadata.get(
            vector_id
        )


        if metadata is None:

            continue


        chunk_id = metadata.get(
            "chunk_id"
        )


        text = chunk_text_map.get(
            chunk_id
        )


        if text is None:

            continue


        results.append(
            {
                "rank": rank,
                "score": float(score),
                "vector_id": vector_id,
                "chunk_id": chunk_id,
                "document_id":
                    metadata.get(
                        "document_id"
                    ),
                "source":
                    metadata.get(
                        "source"
                    ),
                "title":
                    metadata.get(
                        "title"
                    ),
                "path":
                    metadata.get(
                        "path"
                    ),
                "text": text,
            }
        )


    # =========================================================================
    # TOTAL LATENCY
    # =========================================================================

    total_time = (
        embedding_time
        + search_time
    )


    return {
        "query": query,
        "k": k,
        "results": results,
        "timing": {
            "query_embedding_sec":
                embedding_time,
            "faiss_search_sec":
                search_time,
            "total_sec":
                total_time,
        },
    }

### **Initialize Retriever**

In [11]:
FAISS_DIR = Path(
    cliffordimaguezegie_telecom_bge_m3_faiss_path
)

CHUNK_DIR = Path(
    cliffordimaguezegie_telecom_chunks_path
)

retriever_status = initialize_retriever(
    faiss_dir=FAISS_DIR,
    chunk_dir=CHUNK_DIR,
    device="cuda",
)

print("=" * 90)
print("RETRIEVER V1 INITIALIZED")
print("=" * 90)

for key, value in retriever_status.items():
    print(
        f"{key:<25}: {value}"
    )

print("=" * 90)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

RETRIEVER V1 INITIALIZED
version                  : V1
model                    : BAAI/bge-m3
device                   : cuda
embedding_dimension      : 1024
faiss_vectors            : 1506367
faiss_index              : IndexFlatIP
similarity               : inner_product
metadata_rows            : 1506367
chunk_rows               : 1506367
default_k                : 7


### **Validate Retriever**

In [12]:
# =============================================================================
# RAG V1 — RETRIEVER V1 FINAL VALIDATION
# =============================================================================

TEST_QUERY = (
    "What are the primary responsibilities of the AMF "
    "in a 5G Standalone network?"
)

results = retrieve(
    TEST_QUERY,
    k=7,
)

print("=" * 90)
print("RAG V1 — RETRIEVER V1 FINAL VALIDATION")
print("=" * 90)

print(
    f"Query              : "
    f"{results['query']}"
)

print(
    f"K                  : "
    f"{results['k']}"
)

print(
    f"Results returned   : "
    f"{len(results['results'])}"
)

print(
    f"Query embedding    : "
    f"{results['timing']['query_embedding_sec']:.4f} sec"
)

print(
    f"FAISS search       : "
    f"{results['timing']['faiss_search_sec']:.4f} sec"
)

print(
    f"Total retrieval    : "
    f"{results['timing']['total_sec']:.4f} sec"
)

if len(results["results"]) != 7:
    raise RuntimeError(
        "Retriever V1 did not return 7 results."
    )

top = results["results"][0]

print("\nTop result:")

print(
    f"Rank       : {top['rank']}"
)

print(
    f"Score      : {top['score']:.4f}"
)

print(
    f"Chunk ID   : {top['chunk_id']}"
)

print(
    f"Source     : {top['source']}"
)

print(
    f"Title      : {top['title']}"
)

print(
    f"Text       : "
    f"{top['text'][:500]}"
)

print("\n" + "=" * 90)
print("RETRIEVER V1 FINAL VALIDATION PASSED")
print("=" * 90)

RAG V1 — RETRIEVER V1 FINAL VALIDATION
Query              : What are the primary responsibilities of the AMF in a 5G Standalone network?
K                  : 7
Results returned   : 7
Query embedding    : 0.8704 sec
FAISS search       : 0.2586 sec
Total retrieval    : 1.1291 sec

Top result:
Rank       : 1
Score      : 0.7065
Chunk ID   : standards/3gpp_rel18/original/rel_15.docx::chunk_0013
Source     : standards
Title      : rel_15
Text       : the Core Network side, the AMF ("Access and Mobility management Function") oversees all the signalling which is not specific to User Data, such as mobility or security. The SMF ("Session Management Function"), takes care of the signalling related to User Data traffic, such as session establishment. Finally, The UPF ("User Plane Function") represents the handling of user data.
On the Access Network side, the gNB (5G Node B) performs all the main AN-related tasks, including Radio Resource Manageme

RETRIEVER V1 FINAL VALIDATION PASSED


#### **RAG V1 — Retriever V1 Final Validation Observation**

- Retriever V1 was successfully initialized and validated in the Colab RAG environment.
- **K=7** returned all 7 expected ranked results.
- The top similarity score remained **0.7065**, matching the original V1 baseline.
- The same AMF source chunk was retrieved as the top result.
- Total retrieval latency was **1.2811 sec**, including **0.8876 sec** query embedding and **0.3935 sec** FAISS search.

**Status:** Retriever V1 successfully integrated and validated. Ready for General LLM generation.

## **1.3. Load General LLM**

In [13]:
# =============================================================================
# RAG V1 — LOAD GENERAL LLM
# =============================================================================

import time
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)


# =============================================================================
# GENERAL LLM CONFIGURATION
# =============================================================================

GENERAL_LLM_NAME = (
    "EssentialAI/rnj-1-instruct"
)

GENERAL_TEMPERATURE = 0.0
GENERAL_TOP_K = 50
GENERAL_TOP_P = 0.95

GENERAL_MAX_NEW_TOKENS = 512

GENERAL_QUANT_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


# =============================================================================
# GPU STATUS
# =============================================================================

print("=" * 90)
print("RAG V1 — GENERAL LLM")
print("=" * 90)

print(
    f"GPU             : "
    f"{torch.cuda.get_device_name(0)}"
)

print(
    f"GPU memory      : "
    f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
)


# =============================================================================
# LOAD TOKENIZER
# =============================================================================

print("\nLoading General LLM tokenizer...")

tokenizer_start = time.time()

general_tokenizer = (
    AutoTokenizer.from_pretrained(
        GENERAL_LLM_NAME,
        trust_remote_code=True,
    )
)

tokenizer_elapsed = (
    time.time()
    - tokenizer_start
)


# =============================================================================
# LOAD MODEL
# =============================================================================

print(
    "\nLoading General LLM model..."
)

model_start = time.time()

general_model = (
    AutoModelForCausalLM.from_pretrained(
        GENERAL_LLM_NAME,
        quantization_config=GENERAL_QUANT_CONFIG,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
)

general_model.eval()

model_elapsed = (
    time.time()
    - model_start
)


# =============================================================================
# VALIDATION
# =============================================================================

print("\nConfiguration")
print("-" * 60)

print(
    f"Model            : "
    f"{GENERAL_LLM_NAME}"
)

print(
    f"Quantization     : 4-bit NF4"
)

print(
    f"Compute dtype    : float16"
)

print(
    f"Temperature      : "
    f"{GENERAL_TEMPERATURE}"
)

print(
    f"Top-k            : "
    f"{GENERAL_TOP_K}"
)

print(
    f"Top-p            : "
    f"{GENERAL_TOP_P}"
)

print(
    f"Max new tokens   : "
    f"{GENERAL_MAX_NEW_TOKENS}"
)

print(
    f"Tokenizer time   : "
    f"{tokenizer_elapsed:.2f} sec"
)

print(
    f"Model load time  : "
    f"{model_elapsed:.2f} sec"
)

print(
    f"\nDevice map       : "
    f"{getattr(general_model, 'hf_device_map', 'N/A')}"
)


# =============================================================================
# FINAL STATUS
# =============================================================================

print("\n" + "=" * 90)
print("GENERAL LLM LOADED SUCCESSFULLY")
print("=" * 90)

RAG V1 — GENERAL LLM
GPU             : NVIDIA L4
GPU memory      : 22.03 GB

Loading General LLM tokenizer...


config.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'full_attention', 'sliding_attention'}


tokenizer_config.json:   0%|          | 0.00/55.9k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/418 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/226 [00:00<?, ?B/s]


Configuration
------------------------------------------------------------
Model            : EssentialAI/rnj-1-instruct
Quantization     : 4-bit NF4
Compute dtype    : float16
Temperature      : 0.0
Top-k            : 50
Top-p            : 0.95
Max new tokens   : 512
Tokenizer time   : 7.43 sec
Model load time  : 252.30 sec

Device map       : N/A

GENERAL LLM LOADED SUCCESSFULLY


### **1.4. Post GPU Verification**

In [14]:
# =============================================================================
# RAG V1 — CPU / GPU MEMORY STATUS
# =============================================================================

import os
import psutil
import torch


# =============================================================================
# CPU / RAM
# =============================================================================

process = psutil.Process(os.getpid())

system_ram = psutil.virtual_memory()

process_ram_gb = (
    process.memory_info().rss / (1024**3)
)

total_ram_gb = (
    system_ram.total / (1024**3)
)

available_ram_gb = (
    system_ram.available / (1024**3)
)

used_ram_gb = (
    system_ram.used / (1024**3)
)


print("=" * 90)
print("RAG V1 — CPU / GPU MEMORY STATUS")
print("=" * 90)

print("\nCPU / SYSTEM RAM")
print("-" * 40)

print(
    f"Total RAM       : {total_ram_gb:.2f} GB"
)

print(
    f"Used RAM        : {used_ram_gb:.2f} GB"
)

print(
    f"Available RAM   : {available_ram_gb:.2f} GB"
)

print(
    f"Python process   : {process_ram_gb:.2f} GB"
)


# =============================================================================
# GPU
# =============================================================================

print("\nGPU / VRAM")
print("-" * 40)

print(
    f"GPU count       : "
    f"{torch.cuda.device_count()}"
)

for gpu_id in range(
    torch.cuda.device_count()
):

    props = torch.cuda.get_device_properties(
        gpu_id
    )

    allocated_gb = (
        torch.cuda.memory_allocated(gpu_id)
        / (1024**3)
    )

    reserved_gb = (
        torch.cuda.memory_reserved(gpu_id)
        / (1024**3)
    )

    total_gpu_gb = (
        props.total_memory
        / (1024**3)
    )

    free_gpu_gb = (
        total_gpu_gb
        - reserved_gb
    )

    print(
        f"\nGPU {gpu_id}: "
        f"{props.name}"
    )

    print(
        f"  Total VRAM     : "
        f"{total_gpu_gb:.2f} GB"
    )

    print(
        f"  Allocated VRAM : "
        f"{allocated_gb:.2f} GB"
    )

    print(
        f"  Reserved VRAM  : "
        f"{reserved_gb:.2f} GB"
    )

    print(
        f"  Approx. free   : "
        f"{free_gpu_gb:.2f} GB"
    )


print("\n" + "=" * 90)

RAG V1 — CPU / GPU MEMORY STATUS

CPU / SYSTEM RAM
----------------------------------------
Total RAM       : 52.96 GB
Used RAM        : 16.53 GB
Available RAM   : 35.79 GB
Python process   : 14.19 GB

GPU / VRAM
----------------------------------------
GPU count       : 1

GPU 0: NVIDIA L4
  Total VRAM     : 22.03 GB
  Allocated VRAM : 6.77 GB
  Reserved VRAM  : 7.03 GB
  Approx. free   : 15.01 GB



# **2. Context Assembly**

In [15]:
# =============================================================================
# RAG V1 — CONTEXT ASSEMBLY (REFACTORED)
# =============================================================================

def assemble_context(retrieval_results: list) -> str:
    """
    Convert Retriever V1 results into an LLM-ready context block.

    Preserves retrieval rank, source, title, chunk ID, and text
    while remaining resilient to missing metadata.
    """
    if not retrieval_results:
        return "NO RELEVANT CONTEXT FOUND."

    context_blocks = []

    for idx, result in enumerate(retrieval_results, start=1):
        # Extract metadata safely to prevent KeyError
        rank = result.get("rank", idx)
        source = result.get("source", "Unknown Source")
        title = result.get("title", "Untitled Document")
        chunk_id = result.get("chunk_id", "N/A")
        text = result.get("text", "").strip()

        # Clean block structure optimized for LLM attention
        block = (
            f"--- START DOCUMENT [{rank}] ---\n"
            f"Title: {title}\n"
            f"Source: {source}\n"
            f"Chunk ID: {chunk_id}\n\n"
            f"{text}\n"
            f"--- END DOCUMENT [{rank}] ---"
        )
        context_blocks.append(block)

    return "\n\n".join(context_blocks)


# =============================================================================
# TEST CONTEXT ASSEMBLY
# =============================================================================

TEST_QUERY = "What are the primary responsibilities of the AMF in a 5G Standalone network?"

retrieval_output = retrieve(TEST_QUERY, k=7)
rag_context = assemble_context(retrieval_output.get("results", []))

# =============================================================================
# VALIDATION
# =============================================================================

print("=" * 90)
print("RAG V1 — CONTEXT ASSEMBLY VALIDATION")
print("=" * 90)

print(f"Query              : {retrieval_output.get('query', TEST_QUERY)}")
print(f"Retrieved chunks   : {len(retrieval_output.get('results', []))}")
print(f"Context characters : {len(rag_context):,}")

print("\nContext preview:")
print("-" * 90)
print(rag_context[:3000])

print("\n" + "=" * 90)
print("CONTEXT ASSEMBLY VALIDATED")
print("=" * 90)

RAG V1 — CONTEXT ASSEMBLY VALIDATION
Query              : What are the primary responsibilities of the AMF in a 5G Standalone network?
Retrieved chunks   : 7
Context characters : 20,665

Context preview:
------------------------------------------------------------------------------------------
--- START DOCUMENT [1] ---
Title: rel_15
Source: standards
Chunk ID: standards/3gpp_rel18/original/rel_15.docx::chunk_0013

the Core Network side, the AMF ("Access and Mobility management Function") oversees all the signalling which is not specific to User Data, such as mobility or security. The SMF ("Session Management Function"), takes care of the signalling related to User Data traffic, such as session establishment. Finally, The UPF ("User Plane Function") represents the handling of user data.
On the Access Network side, the gNB (5G Node B) performs all the main AN-related tasks, including Radio Resource Management: Radio Bearer Control, Radio Admission Control, Connection Mobility Control, D

# **3. Prompt Construction**

### **Prompt Definition**

In [16]:
# =============================================================================
# TELECOM RAG V1 — HARDENED SYSTEM PROMPT
# =============================================================================

RAG_SYSTEM_PROMPT = """
You are an expert telecom engineering assistant.

Your task is to answer the user's question using the provided telecom document context.

Rules:
1. Base the answer EXCLUSIVELY on the provided context. Do not invent, assume, or extrapolate technical facts beyond what is written.
2. Do not use outside knowledge to fill in missing details.
3. If the context does not contain enough information to answer the question, respond EXACTLY with: "The requested details are not available in the documentation."
4. Answer only what is directly relevant to the user's question.
5. Do not restate or repeat the user's question.
6. Avoid repetition or rephrasing the same technical points.
7. Organize multi-step functions, responsibilities, or architectural components using clear bullet points or short structured sub-headings.
8. Provide only the final answer—do not reveal internal reasoning, scratchpads, or chain-of-thought.
9. Do not mention "context", "documents", "retrieval", "sources", or "prompts".
10. NEVER use meta-introductory phrases such as "Based on the provided context", "According to the documentation", or "The text states".
""".strip()

# =============================================================================
# RAG CHAT MESSAGE BUILDER
# =============================================================================

def build_rag_messages(
    query: str,
    context: str,
) -> list[dict[str, str]]:
    """
    Build standardized chat messages for RAG generation.

    Handles empty or whitespace-only context gracefully without crashing.
    """
    if not isinstance(query, str):
        raise TypeError("query must be a string.")

    if not isinstance(context, str):
        raise TypeError("context must be a string.")

    clean_query = query.strip()
    clean_context = context.strip()

    if not clean_query:
        raise ValueError("query cannot be empty or whitespace-only.")

    # Gracefully handle empty retrieval results without raising an exception
    if not clean_context:
        clean_context = "NO CONTEXT AVAILABLE."

    user_content = f"""
### Context

{clean_context}

### Question

{clean_query}

Provide a concise, technically accurate answer following all system rules.
""".strip()

    return [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_content,
        },
    ]

### **Prompt Validation**

In [17]:
# =============================================================================
# RAG V1 — PROMPT MESSAGE VALIDATION (ENHANCED)
# =============================================================================

# 1. Execute Pipeline Steps Safely
retrieval_output = retrieve(TEST_QUERY, k=7)
retrieved_chunks = retrieval_output.get("results", [])

rag_context = assemble_context(retrieved_chunks)

rag_messages = build_rag_messages(
    query=TEST_QUERY,
    context=rag_context,
)

# 2. Extract Message Metrics
system_msg = rag_messages[0]["content"]
user_msg = rag_messages[1]["content"]

# Standard rule of thumb: ~4 characters per token for English text
est_user_tokens = len(user_msg) // 4
est_system_tokens = len(system_msg) // 4
est_total_tokens = est_user_tokens + est_system_tokens

# 3. Print Validation Telemetry
print("=" * 90)
print("RAG V1 — RAG MESSAGE VALIDATION")
print("=" * 90)

print(f"Query              : {TEST_QUERY}")
print(f"Retrieved Chunks   : {len(retrieved_chunks)}")
print(f"Context Characters : {len(rag_context):,}")
print(f"Message Count      : {len(rag_messages)}")
print(f"Est. Total Tokens  : ~{est_total_tokens:,} (System: ~{est_system_tokens}, User: ~{est_user_tokens})")

print("\nSystem Message:")
print("-" * 90)
print(system_msg)

print("\nUser Message Preview (First 3,000 chars):")
print("-" * 90)
print(user_msg[:3000])

if len(user_msg) > 3000:
    print(f"\n... [Truncated {len(user_msg) - 3000:,} remaining characters from preview]")

print("\n" + "=" * 90)
print("RAG MESSAGE VALIDATION COMPLETE")
print("=" * 90)

RAG V1 — RAG MESSAGE VALIDATION
Query              : What are the primary responsibilities of the AMF in a 5G Standalone network?
Retrieved Chunks   : 7
Context Characters : 20,665
Message Count      : 2
Est. Total Tokens  : ~5,495 (System: ~284, User: ~5211)

System Message:
------------------------------------------------------------------------------------------
You are an expert telecom engineering assistant.

Your task is to answer the user's question using the provided telecom document context.

Rules:
1. Base the answer EXCLUSIVELY on the provided context. Do not invent, assume, or extrapolate technical facts beyond what is written.
2. Do not use outside knowledge to fill in missing details.
3. If the context does not contain enough information to answer the question, respond EXACTLY with: "The requested details are not available in the documentation."
4. Answer only what is directly relevant to the user's question.
5. Do not restate or repeat the user's question.
6. Avoid repet

### **Model-Native Chat Template Validation**

In [18]:
# =============================================================================
# RAG V1 — GENERAL LLM CHAT TEMPLATE VALIDATION (FIXED)
# =============================================================================

# 1. Render String Prompt (For logging and length verification)
general_rendered_prompt = general_tokenizer.apply_chat_template(
    rag_messages,
    tokenize=False,
    add_generation_prompt=True,
)

# 2. Tokenize Directly via Chat Template
general_encoded = general_tokenizer.apply_chat_template(
    rag_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
)

# Safely extract tensor input_ids to check shape/ndim
if isinstance(general_encoded, torch.Tensor):
    input_ids = general_encoded
elif isinstance(general_encoded, dict) or hasattr(general_encoded, "data"):
    input_ids = general_encoded["input_ids"]
else:
    input_ids = torch.tensor(general_encoded)

# Handle output shape regardless of whether batch dimension is present
if input_ids.ndim == 1:
    general_token_count = input_ids.shape[0]
else:
    general_token_count = input_ids.shape[-1]

# 3. Robust Model Context Limit Fallback
config = getattr(general_model, "config", None)
general_context_limit = None

if config:
    for attr in ["max_position_embeddings", "max_sequence_length", "seq_length", "n_positions"]:
        if hasattr(config, attr) and getattr(config, attr) is not None:
            general_context_limit = getattr(config, attr)
            break

# =============================================================================
# VALIDATION OUTPUT & GUARDRAILS
# =============================================================================

print("=" * 90)
print("RAG V1 — GENERAL LLM CHAT TEMPLATE VALIDATION")
print("=" * 90)

print(f"Rendered characters : {len(general_rendered_prompt):,}")
print(f"Input tokens        : {general_token_count:,}")

if general_context_limit:
    utilization = (general_token_count / general_context_limit) * 100
    print(f"Context limit       : {general_context_limit:,}")
    print(f"Context utilization : {utilization:.2f}%")

    # Active Safety Guardrail
    if general_token_count > general_context_limit:
        print("\n[WARNING] Input tokens exceed the maximum context limit!")
        print("Truncate 'k' or reduce chunk size to prevent CUDA OOM.")
    elif utilization > 85.0:
        print("\n[NOTICE] High context utilization. Ensure enough space remains for max_new_tokens.")
else:
    print("Context limit       : Unknown (Config attribute not found)")

print("=" * 90)
print("GENERAL LLM CHAT TEMPLATE VALIDATED")
print("=" * 90)

RAG V1 — GENERAL LLM CHAT TEMPLATE VALIDATION
Rendered characters : 22,209
Input tokens        : 5,119
Context limit       : 32,768
Context utilization : 15.62%
GENERAL LLM CHAT TEMPLATE VALIDATED


# **4. End-to-End General LLM RAG Generation**

## **Generation Configuration Definition**

In [19]:
# =============================================================================
# RAG V1 — GENERAL LLM GENERATION CONFIGURATION
# =============================================================================

GENERAL_GENERATION_CONFIG = {
    "temperature": 0.01,
    "top_k": 50,
    "top_p": 0.95,
    "repetition_penalty": 1.05,
    "max_new_tokens": 1024,
    "do_sample": True,
}


print("=" * 90)
print("RAG V1 — GENERAL LLM GENERATION CONFIGURATION")
print("=" * 90)

print(
    f"Temperature        : "
    f"{GENERAL_GENERATION_CONFIG['temperature']}"
)

print(
    f"Top-k              : "
    f"{GENERAL_GENERATION_CONFIG['top_k']}"
)

print(
    f"Top-p              : "
    f"{GENERAL_GENERATION_CONFIG['top_p']}"
)

print(
    f"Repetition penalty : "
    f"{GENERAL_GENERATION_CONFIG['repetition_penalty']}"
)

print(
    f"Max new tokens     : "
    f"{GENERAL_GENERATION_CONFIG['max_new_tokens']}"
)

print(
    f"Do sample          : "
    f"{GENERAL_GENERATION_CONFIG['do_sample']}"
)

print("=" * 90)

RAG V1 — GENERAL LLM GENERATION CONFIGURATION
Temperature        : 0.01
Top-k              : 50
Top-p              : 0.95
Repetition penalty : 1.05
Max new tokens     : 1024
Do sample          : True


## **Generation Function Definition**

In [20]:
import torch

# =============================================================================
# RAG V1 — BASE GENERATION FROM FROZEN RETRIEVAL
# =============================================================================

def generate_rag_from_retrieval(
    retrieval_output: dict,
    model,
    tokenizer,
    generation_config: dict,
) -> dict:
    """
    Generate an answer from a frozen retrieval output payload.
    Safely handles device placement, token slicing, and metadata tracking.
    """
    query = retrieval_output.get("query", "")
    retrieved_chunks = retrieval_output.get("results", [])

    # 1. Context Assembly
    rag_context = assemble_context(retrieved_chunks)

    # 2. Build Chat Messages
    messages = build_rag_messages(query=query, context=rag_context)

    # 3. Apply Native Chat Template
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # 4. Tokenize & Move Tensors to Model Device (GPU/CPU Safe)
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)

    input_token_count = inputs["input_ids"].shape[-1]

    # 5. Execute Model Generation
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **generation_config,
            pad_token_id=tokenizer.eos_token_id  # Prevents EOS/PAD warning
        )

    # 6. Slice Out Only Newly Generated Tokens (Strip the Input Prompt)
    generated_tokens = output_ids[0][input_token_count:]
    generated_answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # 7. Return Full Payload for Benchmark Logging
    return {
        "query": query,
        "answer": generated_answer,
        "retrieval_output": retrieval_output,
        "prompt_tokens": input_token_count,
        "completion_tokens": len(generated_tokens),
        "total_tokens": input_token_count + len(generated_tokens),
    }


# =============================================================================
# RAG V1 — GENERAL LLM RAG GENERATION WRAPPER
# =============================================================================

def generate_rag(
    query: str,
    model,
    tokenizer,
    generation_config: dict,
    k: int = 7,
) -> dict:
    """
    Execute end-to-end Telecom RAG generation using the General LLM.
    Intended for single-query smoke tests and interactive RAG testing.
    """
    # 1. Live Retrieval
    retrieval_output = retrieve(query, k=k)

    # 2. Generate From Retrieved Context
    return generate_rag_from_retrieval(
        retrieval_output=retrieval_output,
        model=model,
        tokenizer=tokenizer,
        generation_config=generation_config,
    )

In [21]:
# =============================================================================
# RAG V1 — GENERAL LLM GENERATION FROM FROZEN RETRIEVAL (HARDENED)
# =============================================================================

def generate_rag_from_retrieval(
    retrieval_output: dict,
    model,
    tokenizer,
    generation_config: dict,
) -> dict:
    """
    Generate a General LLM Telecom RAG response from precomputed/frozen retrieval results.
    BGE-M3 and FAISS are NOT called here.
    """

    # 1. Context Assembly
    results = retrieval_output.get("results", [])
    context = assemble_context(results)

    # 2. General LLM RAG Messages
    query = retrieval_output.get("query", "")
    messages = build_rag_messages(query, context)

    # 3. Native Chat Template Tokenization
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    # 4. Move Inputs to GPU/Model Device
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    # 5. Determine Padding Token Safely
    pad_token_id = (
        tokenizer.pad_token_id
        if tokenizer.pad_token_id is not None
        else tokenizer.eos_token_id
    )

    # 6. Model Generation
    generation_start = time.time()

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            **generation_config,
            pad_token_id=pad_token_id,
        )

    generation_time = time.time() - generation_start

    # 7. Decode Only Generated Tokens
    input_token_count = inputs["input_ids"].shape[-1]
    generated_ids = output_ids[0, input_token_count:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()

    # 8. Return Comprehensive Telemetry Payload
    return {
        "query": query,
        "k": retrieval_output.get("k", len(results)),
        "retrieval": retrieval_output,
        "context": context,
        "messages": messages,
        "answer": answer,
        "generation_time_sec": generation_time,
        "input_tokens": int(input_token_count),
        "output_tokens": int(len(generated_ids)),
        "generation_config": generation_config,
    }

## **Pilot - LLM + RAG Response**

In [22]:
# =============================================================================
# RAG V1 — GENERAL LLM END-TO-END TEST
# =============================================================================

print("=" * 90)
print("RAG V1 — GENERAL LLM END-TO-END TEST")
print("=" * 90)

print(f"Test Query: {TEST_QUERY}\n")
print("Running General LLM RAG Pipeline...")

try:
    general_rag_result = generate_rag(
        query=TEST_QUERY,
        model=general_model,
        tokenizer=general_tokenizer,
        generation_config=GENERAL_GENERATION_CONFIG,
        k=7,
    )

    # Calculate throughput performance
    gen_time = general_rag_result["generation_time_sec"]
    out_tokens = general_rag_result["output_tokens"]
    throughput = (out_tokens / gen_time) if gen_time > 0 else 0.0

    # =========================================================================
    # SUMMARY TELEMETRY
    # =========================================================================

    print(
        f"\nGeneral LLM complete | "
        f"Input tokens: {general_rag_result['input_tokens']:,} | "
        f"Output tokens: {out_tokens:,} | "
        f"Time: {gen_time:.2f}s | "
        f"Speed: {throughput:.2f} tok/s"
    )

    # =========================================================================
    # DISPLAY RESPONSE
    # =========================================================================

    print("\n" + "=" * 90)
    print("GENERAL LLM RAG ANSWER")
    print("=" * 90)
    print(general_rag_result["answer"])
    print("\n" + "=" * 90)
    print("GENERAL LLM RAG TEST COMPLETE")
    print("=" * 90)

except Exception as e:
    print(f"\n[ERROR] Pipeline execution failed: {str(e)}")

RAG V1 — GENERAL LLM END-TO-END TEST
Test Query: What are the primary responsibilities of the AMF in a 5G Standalone network?

Running General LLM RAG Pipeline...

General LLM complete | Input tokens: 5,119 | Output tokens: 165 | Time: 28.47s | Speed: 5.80 tok/s

GENERAL LLM RAG ANSWER
The primary responsibilities of the AMF (Access and Mobility Management Function) in a 5G Standalone network include:

- Performing Non-Access Stratum (NAS) signaling termination and NAS signaling security.
- Managing Access Stratum (AS) Security control.
- Facilitating inter-CN node signaling for mobility between 3GPP access networks.
- Handling Idle mode UE Reachability, including control and execution of paging retransmission.
- Managing Registration Area and supporting both intra-system and inter-system mobility.
- Conducting Access Authentication and Authorization, including checking roaming rights.
- Controlling mobility management (subscription and policies).
- Supporting Network Slicing and SMF s

# **5. Telecom Benchmark Evaluation**

## **General LLM RAG Inference**

### **Track 1 Inference**

In [23]:
import gc
import json
import time
from datetime import datetime, timezone
import torch

# =============================================================================
# HELPER: AGGRESSIVE HARDWARE SYNCHRONIZED MEMORY CLEARING
# =============================================================================

def flush_vram(delay_sec: float = 0.5):
    """
    Forces garbage collection, releases cached CUDA VRAM back to the GPU,
    synchronizes CPU/GPU threads, and introduces a brief settlement delay.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()  # Block CPU until GPU cache release finishes
    if delay_sec > 0:
        time.sleep(delay_sec)


# =============================================================================
# CONFIGURATION
# =============================================================================

TRACK1_K = 7
OUTPUT_FILE = f"track1_general_rag_results_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.json"

track1_general_results = []
track1_start = time.time()

# Initial global cleanup before benchmark execution
flush_vram(delay_sec=1.0)


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("RAG V1 — TRACK 1 | GENERAL LLM + TELECOM RAG")
print("LIVE RETRIEVAL")
print("=" * 90)

print(f"Questions       : {len(benchmark_questions)}")
print(f"Retriever K     : {TRACK1_K}")
print("=" * 90)


# =============================================================================
# EXECUTE ALL QUESTIONS
# =============================================================================

for index, item in enumerate(benchmark_questions, start=1):

    # 1. PRE-QUERY CLEANUP & SYNCHRONIZATION
    flush_vram(delay_sec=0.5)

    question_id = item["id"]
    category = item["category"]
    question = item["question"]

    print(f"\n[{index:02d}/{len(benchmark_questions):02d}] {question_id} | {category}")
    start_time = time.time()

    # =========================================================================
    # INFERENCE (wrapped in torch.inference_mode to disable autograd tracking)
    # =========================================================================
    try:
        with torch.inference_mode():
            result = generate_rag(
                query=question,
                model=general_model,
                tokenizer=general_tokenizer,
                generation_config=GENERAL_GENERATION_CONFIG,
                k=TRACK1_K,
            )

        record = {
            "question_id": question_id,
            "category": category,
            "question": question,
            "expected_points": item.get("expected_points", []),
            "retriever": {
                "version": "V1",
                "k": TRACK1_K,
            },
            "status": "PASS",
            "answer": result["answer"],
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
            "generation_time_sec": result["generation_time_sec"],
            "retrieval": result["retrieval"],
            "generation_config": result["generation_config"],
            "error": None,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        print(
            f"  PASS | "
            f"Input: {result['input_tokens']:,} | "
            f"Output: {result['output_tokens']:,} | "
            f"Time: {result['generation_time_sec']:.2f} sec"
        )

    except Exception as exc:
        elapsed = time.time() - start_time

        # 2. ERROR RECOVERY CLEANUP
        # Immediately clears allocated memory from the failed forward pass
        flush_vram(delay_sec=0.5)

        record = {
            "question_id": question_id,
            "category": category,
            "question": question,
            "expected_points": item.get("expected_points", []),
            "retriever": {
                "version": "V1",
                "k": TRACK1_K,
            },
            "status": "FAIL",
            "answer": None,
            "input_tokens": None,
            "output_tokens": None,
            "generation_time_sec": elapsed,
            "retrieval": None,
            "generation_config": GENERAL_GENERATION_CONFIG,
            "error": f"{type(exc).__name__}: {exc}",
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        print(f"  FAIL | {type(exc).__name__}: {exc}")

    track1_general_results.append(record)

    # Incremental Checkpoint Save
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(track1_general_results, f, indent=2, ensure_ascii=False)

    # 3. POST-QUERY CLEANUP
    flush_vram(delay_sec=0.2)


# =============================================================================
# SUMMARY & TELEMETRY
# =============================================================================

track1_elapsed = time.time() - track1_start
track1_pass = sum(r["status"] == "PASS" for r in track1_general_results)
track1_fail = sum(r["status"] == "FAIL" for r in track1_general_results)

print("\n" + "=" * 90)
print("RAG V1 — TRACK 1 BENCHMARK COMPLETE")
print("=" * 90)

print(f"Expected questions : {len(benchmark_questions)}")
print(f"Captured           : {len(track1_general_results)}")
print(f"PASS               : {track1_pass}")
print(f"FAIL               : {track1_fail}")
print(f"Runtime            : {track1_elapsed / 60:.2f} min")
print(f"Saved payload to   : {OUTPUT_FILE}")
print("=" * 90)

RAG V1 — TRACK 1 | GENERAL LLM + TELECOM RAG
LIVE RETRIEVAL
Questions       : 20
Retriever K     : 7

[01/20] Q01 | 5G Core
  PASS | Input: 5,119 | Output: 219 | Time: 34.24 sec

[02/20] Q02 | 5G Core
  PASS | Input: 5,289 | Output: 140 | Time: 25.41 sec

[03/20] Q03 | 5G RAN
  PASS | Input: 5,139 | Output: 250 | Time: 38.33 sec

[04/20] Q04 | 5G RAN
  PASS | Input: 5,526 | Output: 323 | Time: 50.24 sec

[05/20] Q05 | 5G SA Procedures
  PASS | Input: 6,137 | Output: 449 | Time: 73.54 sec

[06/20] Q06 | 5G SA Procedures
  PASS | Input: 6,226 | Output: 296 | Time: 52.53 sec

[07/20] Q07 | Open RAN
  PASS | Input: 4,853 | Output: 687 | Time: 89.32 sec

[08/20] Q08 | Open RAN
  PASS | Input: 3,626 | Output: 454 | Time: 47.69 sec

[09/20] Q09 | Cloud-Native Telecom
  PASS | Input: 4,681 | Output: 271 | Time: 37.45 sec

[10/20] Q10 | Cloud-Native Telecom
  PASS | Input: 5,352 | Output: 134 | Time: 24.99 sec

[11/20] Q11 | Applied Telecom Engineering
  PASS | Input: 4,201 | Output: 785 | Time

### **Track 2 Inference**

In [24]:
# =============================================================================
# RAG V1 — TRACK 2 BENCHMARK | GENERAL LLM + TELECOM RAG (OOM-PROTECTED)
# LIVE RETRIEVAL
# =============================================================================

import gc
import json
import time
from datetime import datetime, timezone
import torch


# =============================================================================
# HELPER: AGGRESSIVE HARDWARE SYNCHRONIZED MEMORY CLEARING
# =============================================================================

def flush_vram(delay_sec: float = 0.5):
    """
    Forces garbage collection, releases cached CUDA VRAM back to the GPU,
    synchronizes CPU/GPU threads, and introduces a brief settlement delay.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()  # Block CPU until GPU cache release finishes
    if delay_sec > 0:
        time.sleep(delay_sec)


# =============================================================================
# CONFIGURATION
# =============================================================================

TRACK2_K = 7  # Reduce to 4 or 5 if single-prompt OOMs occur
OUTPUT_FILE = f"track2_general_rag_results_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.json"

track2_general_results = []
track2_start = time.time()

# Initial global cleanup before benchmark start
flush_vram(delay_sec=1.0)


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("RAG V1 — TRACK 2 | GENERAL LLM + TELECOM RAG")
print("LIVE RETRIEVAL")
print("=" * 90)

print(f"Questions       : {len(benchmark_questions_track2)}")
print(f"Retriever K     : {TRACK2_K}")
print("=" * 90)


# =============================================================================
# EXECUTE BENCHMARK
# =============================================================================

for index, item in enumerate(benchmark_questions_track2, start=1):

    # 1. PRE-QUERY CLEANUP & SYNCHRONIZATION
    flush_vram(delay_sec=0.5)

    question_id = item["evaluation_id"]
    benchmark = item["benchmark"]
    question = item["question"]
    choices = item.get("choices")

    # Format model query without target answer leakage
    if choices:
        choices_text = "\n".join(str(choice) for choice in choices)
        model_query = f"{question}\n\nChoices:\n{choices_text}"
    else:
        model_query = question

    print(f"\n[{index:02d}/{len(benchmark_questions_track2):02d}] {question_id} | {benchmark}")
    start_time = time.time()

    # =========================================================================
    # INFERENCE (wrapped in torch.inference_mode to disable autograd tracking)
    # =========================================================================
    try:
        with torch.inference_mode():
            result = generate_rag(
                query=model_query,
                model=general_model,
                tokenizer=general_tokenizer,
                generation_config=GENERAL_GENERATION_CONFIG,
                k=TRACK2_K,
            )

        record = {
            "question_id": question_id,
            "benchmark": benchmark,
            "question": question,
            "choices": choices,
            "expected_answer": item.get("answer"),
            "explanation": item.get("explanation"),
            "candidate_selection_score": item.get("candidate_selection_score"),
            "retriever": {
                "version": "V1",
                "k": TRACK2_K,
            },
            "status": "PASS",
            "answer": result["answer"],
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
            "generation_time_sec": result["generation_time_sec"],
            "retrieval": result["retrieval"],
            "generation_config": result["generation_config"],
            "error": None,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        print(
            f"  PASS | "
            f"Input: {result['input_tokens']:,} | "
            f"Output: {result['output_tokens']:,} | "
            f"Time: {result['generation_time_sec']:.2f} sec"
        )

    except Exception as exc:
        elapsed = time.time() - start_time

        # 2. ERROR RECOVERY CLEANUP
        # Immediately clears allocated memory from the failed forward pass
        flush_vram(delay_sec=0.5)

        record = {
            "question_id": question_id,
            "benchmark": benchmark,
            "question": question,
            "choices": choices,
            "expected_answer": item.get("answer"),
            "explanation": item.get("explanation"),
            "candidate_selection_score": item.get("candidate_selection_score"),
            "retriever": {
                "version": "V1",
                "k": TRACK2_K,
            },
            "status": "FAIL",
            "answer": None,
            "input_tokens": None,
            "output_tokens": None,
            "generation_time_sec": elapsed,
            "retrieval": None,
            "generation_config": GENERAL_GENERATION_CONFIG,
            "error": f"{type(exc).__name__}: {exc}",
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        print(f"  FAIL | {type(exc).__name__}: {exc}")

    track2_general_results.append(record)

    # Save to disk after every iteration to prevent loss during crashes
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(track2_general_results, f, indent=2, ensure_ascii=False)

    # 3. POST-QUERY CLEANUP
    flush_vram(delay_sec=0.2)


# =============================================================================
# SUMMARY
# =============================================================================

track2_elapsed = time.time() - track2_start
track2_pass = sum(r["status"] == "PASS" for r in track2_general_results)
track2_fail = sum(r["status"] == "FAIL" for r in track2_general_results)

print("\n" + "=" * 90)
print("RAG V1 — TRACK 2 BENCHMARK COMPLETE")
print("=" * 90)

print(f"Expected questions : {len(benchmark_questions_track2)}")
print(f"Captured           : {len(track2_general_results)}")
print(f"PASS               : {track2_pass}")
print(f"FAIL               : {track2_fail}")
print(f"Runtime            : {track2_elapsed / 60:.2f} min")
print(f"Artifacts saved    : {OUTPUT_FILE}")
print("=" * 90)

RAG V1 — TRACK 2 | GENERAL LLM + TELECOM RAG
LIVE RETRIEVAL
Questions       : 32
Retriever K     : 7

[01/32] T2-01 | 3gpp_tsg
  FAIL | OutOfMemoryError: CUDA out of memory. Tried to allocate 1.58 GiB. GPU 0 has a total capacity of 22.03 GiB of which 1.35 GiB is free. Including non-PyTorch memory, this process has 20.67 GiB memory in use. Of the allocated memory 20.30 GiB is allocated by PyTorch, and 147.80 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

[02/32] T2-02 | 3gpp_tsg
  FAIL | OutOfMemoryError: CUDA out of memory. Tried to allocate 7.09 GiB. GPU 0 has a total capacity of 22.03 GiB of which 6.80 GiB is free. Including non-PyTorch memory, this process has 15.23 GiB memory in use. Of the allocated memory 14